# DSAP-34 — Entrenamiento y evaluación de modelos NYC 311

## Objetivo

Entrenar y comparar modelos de clasificación multiclase para
predecir la demanda semanal de solicitudes NYC 311 por borough.

Se utilizarán variables históricas y de calendario, respetando
el orden temporal de los datos.

La evaluación principal se realizará mediante F1-macro y se
registrarán los experimentos en MLflow.

In [3]:
import pandas as pd
from pathlib import Path

# Cargar el dataset preparado en el notebook 04
ruta = Path("../data/processed/dataset_modelado_semanal.parquet")

if not ruta.exists():
    ruta = Path("data/processed/dataset_modelado_semanal.parquet")

datos = pd.read_parquet(ruta)

print("Dataset cargado correctamente")
print("Observaciones:", len(datos))
print("Columnas:", len(datos.columns))

display(datos.head())

Dataset cargado correctamente
Observaciones: 780
Columnas: 17


,borough,semana,total_solicitudes,mes,semana_anio,lag_1,lag_2,lag_4,lag_52,media_movil_4,media_movil_12,media_movil_52,std_movil_4,demanda_relativa,P33,P67,nivel_demanda
0,BRONX,2023-01-02,10994,1,1,12435.0,15324.0,12092.0,12056.0,13279.75,12164.500000,14090.750000,1449.534264,0.780228,0.937281,1.069243,Baja
1,BRONX,2023-01-09,12426,1,2,10994.0,12435.0,13268.0,16275.0,13005.25,12140.916667,14070.326923,1808.862511,0.883135,0.937281,1.069243,Baja
2,BRONX,2023-01-16,11513,1,3,12426.0,10994.0,15324.0,14508.0,12794.75,12169.583333,13996.307692,1817.067486,0.822574,0.937281,1.069243,Baja
3,BRONX,2023-01-23,12055,1,4,11513.0,12426.0,12435.0,14915.0,11842.00,12240.750000,13938.711538,711.816924,0.864858,0.937281,1.069243,Baja
4,BRONX,2023-01-30,14801,1,5,12055.0,11513.0,10994.0,14801.0,11747.00,12360.583333,13883.711538,626.543428,1.066069,0.937281,1.069243,Media


## Separación temporal de los datos

Se utilizarán las observaciones de 2023 y 2024 para el
desarrollo del modelo y las observaciones de 2025 como
conjunto de prueba.

La separación se realiza cronológicamente para evitar
mezclar información futura con los datos de entrenamiento.

Se excluyen las variables utilizadas para construir la
etiqueta, como total_solicitudes, demanda_relativa, P33 y P67.

In [4]:
# Separación temporal y selección de variables

datos["semana"] = pd.to_datetime(datos["semana"])

variables = [
    "borough", "mes", "semana_anio",
    "lag_1", "lag_2", "lag_4", "lag_52",
    "media_movil_4", "media_movil_12",
    "media_movil_52", "std_movil_4"
]

objetivo = "nivel_demanda"

train = datos[datos["semana"].dt.year.isin([2023, 2024])].copy()
test = datos[datos["semana"].dt.year == 2025].copy()

X_train = train[variables]
y_train = train[objetivo]

X_test = test[variables]
y_test = test[objetivo]

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)

print("\nDistribución entrenamiento:")
display(y_train.value_counts())

print("\nDistribución prueba:")
display(y_test.value_counts())

print("\nValores faltantes en predictores:")
display(X_train.isna().sum().to_frame("Nulos"))

Entrenamiento: (525, 11)
Prueba: (255, 11)

Distribución entrenamiento:


nivel_demanda
Baja     175
Media    175
Alta     175
Name: count, dtype: int64


Distribución prueba:


nivel_demanda
Media    98
Baja     87
Alta     70
Name: count, dtype: int64


Valores faltantes en predictores:


,Nulos
borough,0
mes,0
semana_anio,0
lag_1,0
lag_2,0
lag_4,0
lag_52,0
media_movil_4,0
media_movil_12,0
media_movil_52,0


## Entrenamiento inicial y comparación de modelos — DSAP-34

Se entrena un modelo de referencia que predice siempre la clase más
frecuente y un modelo Random Forest que utiliza las variables
históricas, geográficas y de calendario.

Se evalúan mediante F1-macro, accuracy y recall de demanda Alta,
utilizando el conjunto temporal de prueba correspondiente a 2025.

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    classification_report,
    confusion_matrix
)

# Transformación de variables categóricas
preprocesador = ColumnTransformer(
    transformers=[
        ("borough", OneHotEncoder(handle_unknown="ignore"), ["borough"])
    ],
    remainder="passthrough"
)

modelos = {
    "Dummy": DummyClassifier(strategy="most_frequent"),
    "RandomForest": RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )
}

resultados = []

for nombre, estimador in modelos.items():

    modelo = Pipeline([
        ("preprocesamiento", preprocesador),
        ("clasificador", estimador)
    ])

    modelo.fit(X_train, y_train)
    predicciones = modelo.predict(X_test)

    accuracy = accuracy_score(y_test, predicciones)
    f1_macro = f1_score(y_test, predicciones, average="macro")
    recall_alta = recall_score(
        y_test, predicciones,
        labels=["Alta"], average="macro",
        zero_division=0
    )

    resultados.append({
        "modelo": nombre,
        "accuracy": accuracy,
        "f1_macro": f1_macro,
        "recall_alta": recall_alta
    })

    print(f"\n=== {nombre} ===")
    print(classification_report(
        y_test, predicciones,
        labels=["Baja", "Media", "Alta"],
        zero_division=0
    ))

resultados_df = pd.DataFrame(resultados)
display(resultados_df.round(4))


=== Dummy ===
              precision    recall  f1-score   support

        Baja       0.00      0.00      0.00        87
       Media       0.00      0.00      0.00        98
        Alta       0.27      1.00      0.43        70

    accuracy                           0.27       255
   macro avg       0.09      0.33      0.14       255
weighted avg       0.08      0.27      0.12       255


=== RandomForest ===
              precision    recall  f1-score   support

        Baja       0.58      0.62      0.60        87
       Media       0.47      0.38      0.42        98
        Alta       0.52      0.61      0.56        70

    accuracy                           0.53       255
   macro avg       0.52      0.54      0.53       255
weighted avg       0.52      0.53      0.52       255



,modelo,accuracy,f1_macro,recall_alta
0,Dummy,0.2745,0.1436,1.0000
1,RandomForest,0.5255,0.5267,0.6143


In [8]:
import mlflow
import mlflow.sklearn
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, recall_score

# Guardar los experimentos dentro del proyecto
raiz = Path.cwd()
if raiz.name == "notebooks":
    raiz = raiz.parent

tracking_dir = raiz / "mlruns"
tracking_dir.mkdir(parents=True, exist_ok=True)

# Utilizar SQLite para almacenar los experimentos
base_datos = raiz / "mlflow.db"

mlflow.set_tracking_uri(
    f"sqlite:///{base_datos.as_posix()}"
)

mlflow.set_experiment(
    "DSAP34_NYC311_Clasificacion_Demanda"
)

for nombre, estimador in modelos.items():

    modelo = Pipeline([
        ("preprocesamiento", preprocesador),
        ("clasificador", estimador)
    ])

    modelo.fit(X_train, y_train)
    predicciones = modelo.predict(X_test)

    metricas = {
        "accuracy": accuracy_score(y_test, predicciones),
        "f1_macro": f1_score(
            y_test, predicciones, average="macro"
        ),
        "recall_alta": recall_score(
            y_test == "Alta",
            predicciones == "Alta",
            zero_division=0
        )
    }

    with mlflow.start_run(run_name=nombre):
        mlflow.log_param("modelo", nombre)
        mlflow.log_param("train_periodo", "2023-2024")
        mlflow.log_param("test_periodo", "2025")
        mlflow.log_param("n_train", len(X_train))
        mlflow.log_param("n_test", len(X_test))

        if nombre == "RandomForest":
            mlflow.log_params({
                "n_estimators": 200,
                "max_depth": 10,
                "random_state": 42
            })

        mlflow.log_metrics(metricas)
        mlflow.sklearn.log_model(
    modelo,
    name="modelo",
    serialization_format="skops",
    skops_trusted_types=["sklearn.tree._tree.Tree"]
    )

    print(f"{nombre} registrado: {metricas}")

print("\nExperimentos guardados en:", tracking_dir)
print("Registro completado.")

Dummy registrado: {'accuracy': 0.27450980392156865, 'f1_macro': 0.1435897435897436, 'recall_alta': 1.0}
RandomForest registrado: {'accuracy': 0.5254901960784314, 'f1_macro': 0.5267235331043905, 'recall_alta': 0.6142857142857143}

Experimentos guardados en: c:\Users\georg\Documents\Uniandes\DSA\dsa-proyecto\mlruns
Registro completado.


## Comparación de modelos con validación temporal

Se comparan tres algoritmos de clasificación supervisada:

- Regresión Logística
- Random Forest
- HistGradientBoosting

Para seleccionar el modelo se utiliza 2023 como período de
entrenamiento y 2024 como período de validación.

Los umbrales de clasificación de esta validación se calculan
exclusivamente con datos de 2023. De esta forma, las etiquetas
de 2024 no dependen de información utilizada posteriormente
para evaluar los modelos.

La métrica principal es F1-macro. También se analizan accuracy
y recall de demanda Alta.

El año 2025 se reserva para la evaluación final.

In [9]:
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier
)
from sklearn.metrics import accuracy_score, f1_score, recall_score

# 1. Separación temporal para seleccionar el modelo
desarrollo = datos[
    datos["semana"].dt.year.isin([2023, 2024])
].copy()

calibracion = desarrollo[
    desarrollo["semana"].dt.year == 2023
].copy()

validacion = desarrollo[
    desarrollo["semana"].dt.year == 2024
].copy()

# 2. Calcular umbrales únicamente con 2023
umbrales_2023 = (
    calibracion.groupby("borough")["demanda_relativa"]
    .quantile([0.33, 0.67])
    .unstack()
    .rename(columns={0.33: "p33_val", 0.67: "p67_val"})
)

desarrollo = desarrollo.merge(
    umbrales_2023, on="borough", how="left"
)

desarrollo["objetivo_validacion"] = np.select(
    [
        desarrollo["demanda_relativa"] < desarrollo["p33_val"],
        desarrollo["demanda_relativa"] <= desarrollo["p67_val"]
    ],
    ["Baja", "Media"],
    default="Alta"
)

train_val = desarrollo[
    desarrollo["semana"].dt.year == 2023
].copy()

test_val = desarrollo[
    desarrollo["semana"].dt.year == 2024
].copy()

X_train_val = train_val[variables]
y_train_val = train_val["objetivo_validacion"]

X_test_val = test_val[variables]
y_test_val = test_val["objetivo_validacion"]

# 3. Preprocesamiento
numericas = [c for c in variables if c != "borough"]

preprocesador_val = ColumnTransformer([
    ("categoria",
     OneHotEncoder(handle_unknown="ignore", sparse_output=False),
     ["borough"]),
    ("numericas", StandardScaler(), numericas)
])

# 4. Modelos candidatos
candidatos = {
    "RegresionLogistica": LogisticRegression(
        max_iter=2000,
        random_state=42
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    ),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        max_iter=200,
        max_leaf_nodes=15,
        learning_rate=0.05,
        random_state=42
    )
}

# 5. Entrenar y comparar
resultados_validacion = []

for nombre, estimador in candidatos.items():

    pipeline = Pipeline([
        ("preprocesamiento", preprocesador_val),
        ("clasificador", estimador)
    ])

    pipeline.fit(X_train_val, y_train_val)
    pred = pipeline.predict(X_test_val)

    metricas = {
        "modelo": nombre,
        "accuracy": accuracy_score(y_test_val, pred),
        "f1_macro": f1_score(
            y_test_val, pred, average="macro"
        ),
        "recall_alta": recall_score(
            y_test_val == "Alta",
            pred == "Alta",
            zero_division=0
        )
    }

    resultados_validacion.append(metricas)

resultados_validacion_df = (
    pd.DataFrame(resultados_validacion)
    .sort_values("f1_macro", ascending=False)
    .reset_index(drop=True)
)

print("Entrenamiento 2023:", len(X_train_val))
print("Validación 2024:", len(X_test_val))

print("\nComparación de modelos:")
display(resultados_validacion_df.round(4))

print(
    "\nMejor candidato:",
    resultados_validacion_df.loc[0, "modelo"]
)

Entrenamiento 2023: 260
Validación 2024: 265

Comparación de modelos:


,modelo,accuracy,f1_macro,recall_alta
0,RegresionLogistica,0.5472,0.5346,0.6267
1,RandomForest,0.5019,0.4971,0.6267
2,HistGradientBoosting,0.4604,0.4569,0.5067



Mejor candidato: RegresionLogistica


In [10]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import RidgeClassifier, LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.metrics import accuracy_score, f1_score, recall_score

modelos_simples = {
    "Dummy": DummyClassifier(strategy="most_frequent"),

    "RegresionLogistica": LogisticRegression(
        max_iter=2000,
        random_state=42
    ),

    "Arbol_Depth3": DecisionTreeClassifier(
        max_depth=3,
        min_samples_leaf=10,
        random_state=42
    ),

    "Arbol_Depth5": DecisionTreeClassifier(
        max_depth=5,
        min_samples_leaf=10,
        random_state=42
    ),

    "NaiveBayes": GaussianNB(),

    "RidgeClassifier": RidgeClassifier(alpha=1.0)
}

resultados_simples = []

for nombre, clasificador in modelos_simples.items():

    modelo = Pipeline([
        ("preprocesamiento", clone(preprocesador_val)),
        ("clasificador", clasificador)
    ])

    modelo.fit(X_train_val, y_train_val)
    pred = modelo.predict(X_test_val)

    resultados_simples.append({
        "modelo": nombre,
        "accuracy": accuracy_score(y_test_val, pred),
        "f1_macro": f1_score(
            y_test_val, pred, average="macro"
        ),
        "recall_alta": recall_score(
            y_test_val == "Alta",
            pred == "Alta",
            zero_division=0
        )
    })

comparacion_simples = (
    pd.DataFrame(resultados_simples)
    .sort_values("f1_macro", ascending=False)
    .reset_index(drop=True)
)

display(comparacion_simples.round(4))

,modelo,accuracy,f1_macro,recall_alta
0,RidgeClassifier,0.5887,0.5561,0.7200
1,RegresionLogistica,0.5472,0.5346,0.6267
2,Arbol_Depth3,0.4302,0.4237,0.6400
3,NaiveBayes,0.3925,0.3857,0.4933
4,Arbol_Depth5,0.3660,0.3484,0.4133
5,Dummy,0.3321,0.1662,0.0000


In [11]:
# opctimizar rigde

from sklearn.linear_model import RidgeClassifier
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.metrics import accuracy_score, f1_score, recall_score
import pandas as pd

resultados_ridge = []

for alpha in [0.01, 0.1, 1.0, 10.0, 100.0]:

    modelo_ridge = Pipeline([
        ("preprocesamiento", clone(preprocesador_val)),
        ("clasificador", RidgeClassifier(alpha=alpha))
    ])

    modelo_ridge.fit(X_train_val, y_train_val)
    pred = modelo_ridge.predict(X_test_val)

    resultados_ridge.append({
        "alpha": alpha,
        "accuracy": accuracy_score(y_test_val, pred),
        "f1_macro": f1_score(
            y_test_val, pred, average="macro"
        ),
        "recall_alta": recall_score(
            y_test_val == "Alta",
            pred == "Alta",
            zero_division=0
        )
    })

comparacion_ridge = (
    pd.DataFrame(resultados_ridge)
    .sort_values("f1_macro", ascending=False)
    .reset_index(drop=True)
)

display(comparacion_ridge.round(4))

print(
    "Mejor alpha:",
    comparacion_ridge.iloc[0]["alpha"]
)

,alpha,accuracy,f1_macro,recall_alta
0,0.10,0.6000,0.5900,0.6533
1,0.01,0.5849,0.5784,0.6267
2,1.00,0.5887,0.5561,0.7200
3,10.00,0.4755,0.4387,0.6800
4,100.00,0.4566,0.4180,0.6133


Mejor alpha: 0.1


## Evaluación final de RidgeClassifier

Se selecciona RidgeClassifier con alpha = 0.1 por obtener el
mayor F1-macro durante la validación temporal de 2024.

El modelo se entrena nuevamente con las observaciones de
2023–2024 y se evalúa sobre 2025.

Se analizan accuracy, F1-macro, recall de demanda Alta
y matriz de confusión para determinar su desempeño fuera
del período de desarrollo.

In [12]:
from sklearn.linear_model import RidgeClassifier
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    classification_report,
    confusion_matrix
)

# Entrenar el modelo seleccionado con 2023-2024
ridge_final = Pipeline([
    ("preprocesamiento", clone(preprocesador_val)),
    ("clasificador", RidgeClassifier(alpha=0.1))
])

ridge_final.fit(X_train, y_train)

# Evaluación sobre 2025
pred_ridge = ridge_final.predict(X_test)

metricas_ridge = {
    "accuracy": accuracy_score(y_test, pred_ridge),
    "f1_macro": f1_score(
        y_test, pred_ridge, average="macro"
    ),
    "recall_alta": recall_score(
        y_test == "Alta",
        pred_ridge == "Alta",
        zero_division=0
    )
}

print("=== RIDGE FINAL — EVALUACIÓN 2025 ===")
display(pd.DataFrame([metricas_ridge]).round(4))

print("\nReporte de clasificación:")
print(classification_report(
    y_test, pred_ridge,
    labels=["Baja", "Media", "Alta"],
    zero_division=0
))

print("\nMatriz de confusión:")
display(pd.DataFrame(
    confusion_matrix(
        y_test, pred_ridge,
        labels=["Baja", "Media", "Alta"]
    ),
    index=["Real Baja", "Real Media", "Real Alta"],
    columns=["Pred Baja", "Pred Media", "Pred Alta"]
))

=== RIDGE FINAL — EVALUACIÓN 2025 ===


,accuracy,f1_macro,recall_alta
0,0.5686,0.5386,0.7571



Reporte de clasificación:
              precision    recall  f1-score   support

        Baja       0.52      0.85      0.65        87
       Media       0.64      0.18      0.29        98
        Alta       0.62      0.76      0.68        70

    accuracy                           0.57       255
   macro avg       0.60      0.60      0.54       255
weighted avg       0.60      0.57      0.52       255


Matriz de confusión:


,Pred Baja,Pred Media,Pred Alta
Real Baja,74,5,8
Real Media,56,18,24
Real Alta,12,5,53


In [13]:
import mlflow
import mlflow.sklearn

# Utilizar la configuración SQLite de MLflow ya establecida
with mlflow.start_run(run_name="RidgeClassifier_alpha_0.1"):

    mlflow.log_params({
        "modelo": "RidgeClassifier",
        "alpha": 0.1,
        "train_periodo": "2023-2024",
        "test_periodo": "2025",
        "n_train": len(X_train),
        "n_test": len(X_test),
        "random_state": 42
    })

    mlflow.log_metrics(metricas_ridge)

    mlflow.sklearn.log_model(
        ridge_final,
        name="modelo",
        serialization_format="skops"
    )

print("RidgeClassifier registrado correctamente en MLflow")

RidgeClassifier registrado correctamente en MLflow


## Conclusiones del entrenamiento — DSAP-34

Se compararon modelos de clasificación multiclase para anticipar el
nivel de demanda semanal de solicitudes NYC 311.

Durante la validación temporal de 2024, RidgeClassifier con alpha=0.1
obtuvo el mejor F1-macro (0.5900) y un accuracy del 60.00 %.

Al entrenarlo con información de 2023–2024 y evaluarlo en 2025,
alcanzó un accuracy de 56.86 %, F1-macro de 0.5386 y recall del
75.71 % para demanda Alta.

Aunque RidgeClassifier supera al Random Forest en la evaluación
de 2025, presenta dificultades para reconocer la demanda Media,
con un recall aproximado del 18.37 %.

El modelo queda registrado en MLflow junto con sus parámetros,
métricas y artefactos. Se considera el mejor candidato actual,
aunque todavía no alcanza la meta inicial de F1-macro >= 0.60.

Se recomienda ampliar las variables predictoras y comprobar la
estabilidad temporal antes de su utilización operativa.